In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/organizations/flaredown/flaredown-autoimmune-symptom-tracker/export.csv")

df.head()
df.info()
df.describe()

/tmp/ipykernel_55/1143935430.py:3: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/kaggle/input/datasets/organizations/flaredown/flaredown-autoimmune-symptom-tracker/export.csv")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7976223 entries, 0 to 7976222
Data columns (total 9 columns):
 #   Column           Dtype  
---  ------           -----  
 0   user_id          object 
 1   age              float64
 2   sex              object 
 3   country          object 
 4   checkin_date     object 
 5   trackable_id     object 
 6   trackable_type   object 
 7   trackable_name   object 
 8   trackable_value  object 
dtypes: float64(1), object(8)
memory usage: 547.7+ MB


,age
count,7.666997e+06
mean,3.506967e+01
std,1.437926e+02
min,-1.966910e+05
25%,2.600000e+01
50%,3.400000e+01
75%,4.300000e+01
max,2.018000e+03


In [1]:
df = df.sample(100000, random_state=42)

NameError: name 'df' is not defined

In [4]:
import pandas as pd

df = pd.read_csv(
    "/kaggle/input/datasets/organizations/flaredown/flaredown-autoimmune-symptom-tracker/export.csv",
    low_memory=False
)

In [5]:
df = df.sample(100000, random_state=42)

In [6]:
df.head()
df.shape

(100000, 9)

In [7]:
df['trackable_type'].value_counts()

trackable_type
Symptom      45963
Weather      17269
Condition    14050
Treatment    11357
Food          5842
Tag           5518
HBI              1
Name: count, dtype: int64

In [8]:
symptoms_df = df[df['trackable_type'] == 'Symptom']

In [9]:
symptoms_df.head()
symptoms_df.shape

(45963, 9)

In [10]:
pivot_df = symptoms_df.pivot_table(
    index=['user_id', 'checkin_date'],
    columns='trackable_name',
    values='trackable_value',
    aggfunc='mean'
)

TypeError: agg function failed [how->mean,dtype->object]

In [12]:
symptoms_df = symptoms_df.copy()

symptoms_df['trackable_value'] = pd.to_numeric(
    symptoms_df['trackable_value'], errors='coerce'
)

In [13]:
symptoms_df['trackable_value'].dtype

dtype('int64')

In [16]:
symptoms_df = symptoms_df.dropna(subset=['trackable_value'])

In [17]:
symptoms_df.shape

(45963, 9)

In [18]:
pivot_df = symptoms_df.pivot_table(
    index=['user_id', 'checkin_date'],
    columns='trackable_name',
    values='trackable_value',
    aggfunc='mean'
)

In [19]:
pivot_df = pivot_df.reset_index()
pivot_df = pivot_df.fillna(0)

In [20]:
pivot_df.shape
pivot_df.head()

trackable_name,user_id,checkin_date,Leg knee pain,Limited vision,Rash on legs,Upper spine pain,Worried,mind noise,"""Jelly Belly""","""Kidney"" Pain",...,yellow stool,zings in right foot,zink,zofran,zoning out,Übelkeit,“Head Flu”,“Scrambled eggs” brain,“bruising”,💩
0,QEVuQwEA++z6GMJgxyjYYw0jFdXeDw==,2019-10-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,QEVuQwEA++z6GMJgxyjYYw0jFdXeDw==,2019-10-06,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,QEVuQwEA++z6GMJgxyjYYw0jFdXeDw==,2019-10-20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,QEVuQwEA+/WWv2EpSyctc64BtIuDnQ==,2018-03-06,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,QEVuQwEA+/gllBhhcqr2iAESqNOeUA==,2019-10-14,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [21]:
# count how many non-zero values each column has
non_zero_counts = (pivot_df != 0).sum()

# keep only columns that appear at least 50 times
selected_cols = non_zero_counts[non_zero_counts > 50].index

pivot_df = pivot_df[selected_cols]

In [22]:
pivot_df.shape

(41125, 86)

In [23]:
pivot_df.columns

Index(['user_id', 'checkin_date', 'Abdominal pain', 'Acid Reflux',
       'Ankle pain', 'Anxiety', 'Back pain', 'Bloating', 'Blurred vision',
       'Body aches', 'Body aching', 'Brain fog', 'Chest pain',
       'Chronic Fatigue', 'Chronic pain', 'Constipation', 'Depression',
       'Diarrhea', 'Difficulty concentrating', 'Dizziness', 'Dry eye',
       'Dry eyes', 'Dry mouth', 'Excessive sweating', 'Eye pain', 'Fatigue',
       'Fatigue and tiredness', 'Foot pain', 'Gas', 'Hand pain', 'Headache',
       'Headaches', 'Hip pain', 'Insomnia', 'Irritability', 'Itchy skin',
       'Jaw pain', 'Joint pain', 'Joint stiffness', 'Knee pain', 'Leg pain',
       'Light sensitivity', 'Loss of appetite', 'Low back pain', 'Low mood',
       'Lower Back Pain', 'Memory Problems', 'Menstrual cramps',
       'Mental fatigue', 'Migraine', 'Mood swings', 'Muscle ache',
       'Muscle pain', 'Muscle spasms', 'Muscle twitching', 'Nasal congestion',
       'Nausea', 'Neck pain', 'Night sweats', 'Pain', 'Pelv

In [24]:
target = 'Fatigue'

In [25]:
X = pivot_df.drop(columns=[target])
y = pivot_df[target]

In [27]:
print(X.shape)
print(y.shape)

(41125, 85)
(41125,)


In [28]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [29]:
print(X_train.shape)
print(X_test.shape)

(32900, 85)
(8225, 85)


In [30]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, random_state=42)

model.fit(X_train, y_train)

ValueError: could not convert string to float: 'QEVuQwEAhCiAr51XiKbH2RcJiuPgOw=='

In [31]:
X.dtypes

trackable_name
user_id                object
checkin_date           object
Abdominal pain        float64
Acid Reflux           float64
Ankle pain            float64
                       ...   
exhaustion            float64
heart palpitations    float64
lightheadedness       float64
muscle weakness       float64
nerve pain            float64
Length: 85, dtype: object

In [32]:
X = X.select_dtypes(include=['number'])

In [33]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [34]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [35]:
y_pred = model.predict(X_test)

In [36]:
y_pred[:10]

array([0.        , 0.        , 0.10337035, 0.        , 0.10337035,
       0.10337035, 0.10337035, 0.        , 0.10337035, 0.10337035])

In [37]:
from sklearn.metrics import mean_squared_error

mse = mean_squared_error(y_test, y_pred)
print("MSE:", mse)

MSE: 0.19091377908329468


In [38]:
import numpy as np

rmse = np.sqrt(mse)
print("RMSE:", rmse)

RMSE: 0.43693681360500475


In [39]:
import pandas as pd

importances = model.feature_importances_

feature_importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': importances
})

feature_importance_df = feature_importance_df.sort_values(
    by='importance', ascending=False
)

feature_importance_df.head(10)

,feature,importance
58,Right Hip Pain,0.047687
27,Headache,0.043222
23,Fatigue and tiredness,0.036559
53,Nausea,0.034391
17,Dizziness,0.034340
9,Brain fog,0.025637
42,Lower Back Pain,0.025027
13,Constipation,0.024819
3,Anxiety,0.024747
55,Night sweats,0.022247


In [40]:
input_data = {col: 0 for col in X.columns}

In [42]:
input_data['Diarrhea'] = 1

In [43]:
import pandas as pd

input_df = pd.DataFrame([input_data])

In [44]:
prediction = model.predict(input_df)

print("Predicted Fatigue:", prediction[0])

Predicted Fatigue: 0.011780850865522986


In [47]:
input_data['Headache'] = 2
input_data['Nausea'] = 1

In [48]:
input_df = pd.DataFrame([input_data])

prediction = model.predict(input_df)

print("Predicted Fatigue:", prediction[0])

Predicted Fatigue: 0.0


In [50]:
input_data = {col: 0 for col in X.columns}

input_data['Headache'] = 4
input_data['Nausea'] = 3
input_data['Dizziness'] = 3
input_data['Anxiety'] = 2

input_df = pd.DataFrame([input_data])

prediction = model.predict(input_df)

print("Predicted Fatigue:", prediction[0])

Predicted Fatigue: 0.0


In [51]:
y.value_counts()

Fatigue
0.0    39875
2.0      383
3.0      374
1.0      284
4.0      209
Name: count, dtype: int64

In [52]:
non_zero_df = pivot_df[pivot_df['Fatigue'] > 0]

In [53]:
X_new = non_zero_df.drop(columns=['Fatigue'])
y_new = non_zero_df['Fatigue']

In [54]:
X_new = X_new.select_dtypes(include=['number'])

In [55]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_new, y_new, test_size=0.2, random_state=42
)

In [56]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [57]:
input_data = {col: 0 for col in X.columns}

input_data['Headache'] = 4
input_data['Nausea'] = 3
input_data['Dizziness'] = 3
input_data['Anxiety'] = 2

input_df = pd.DataFrame([input_data])

prediction = model.predict(input_df)

print("Predicted Fatigue:", prediction[0])

Predicted Fatigue: 3.69
